# Індивідуальні завдання до практичної роботи № 3
**Виконав:** Мельниківський Андрій, група КН-41

*Варіант не вказано, тому виконано всі завдання 1–7.*

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Підключення Google Диска (у Colab). Поза Colab — використовується локальна папка.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/'
except ImportError:
    BASE = ''
    print('Colab не виявлено — використовується локальна папка')
PATH = BASE + 'sales.csv'
print('Шлях до файлу:', PATH)

Colab не виявлено — використовується локальна папка
Шлях до файлу: sales.csv


## Завдання 1. Очищення власного «брудного» DataFrame
Створюємо таблицю (5+ колонок) з пропусками, дублікатами та викидами.

In [3]:
emp = pd.DataFrame({
    'id':         [1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10, 10],
    'name':       ['Anna', 'Bohdan', 'Olena', 'Petro', 'Iryna', 'Iryna', 'Oleh', None, 'Maksym', 'Yulia', 'Taras', 'Taras'],
    'department': ['IT', 'IT', 'HR', 'Sales', 'HR', 'HR', 'IT', 'Sales', None, 'IT', 'Sales', 'Sales'],
    'age':        [25, 31, None, 28, 35, 35, 40, 29, 33, 27, 30, 30],
    'experience': [2, 6, 4, None, 10, 10, 15, 4, 8, 3, 5, 5],
    'salary':     [30000, 45000, 28000, 32000, 40000, 40000, 52000, 31000, None, 29000, 500000, 500000]
})
emp

,id,name,department,age,experience,salary
0,1,Anna,IT,25.0,2.0,30000.0
1,2,Bohdan,IT,31.0,6.0,45000.0
2,3,Olena,HR,NaN,4.0,28000.0
3,4,Petro,Sales,28.0,NaN,32000.0
4,5,Iryna,HR,35.0,10.0,40000.0
5,5,Iryna,HR,35.0,10.0,40000.0
6,6,Oleh,IT,40.0,15.0,52000.0
7,7,None,Sales,29.0,4.0,31000.0
8,8,Maksym,None,33.0,8.0,NaN
9,9,Yulia,IT,27.0,3.0,29000.0


In [4]:
print('Пропуски:'); print(emp.isnull().sum())
print('\nПовних дублікатів:', emp.duplicated().sum())

Пропуски:
id            0
name          1
department    1
age           1
experience    1
salary        1
dtype: int64

Повних дублікатів: 2


In [5]:
clean = emp.copy()
# 1) пропуски: медіана для числових, мода для категоріальних
for col in clean.select_dtypes(include='number').columns:
    clean[col] = clean[col].fillna(clean[col].median())
for col in clean.select_dtypes(include='object').columns:
    clean[col] = clean[col].fillna(clean[col].mode()[0])
print('Пропусків після заповнення:', clean.isnull().sum().sum())

# 2) повні дублікати
clean = clean.drop_duplicates()
print('Рядків після видалення дублікатів:', len(clean))

# 3) викиди за IQR (колонка salary)
Q1, Q3 = clean['salary'].quantile(0.25), clean['salary'].quantile(0.75)
IQR = Q3 - Q1
low, up = Q1 - 1.5*IQR, Q3 + 1.5*IQR
print(f'IQR-межі salary: [{low}, {up}]')
clean = clean[(clean['salary'] >= low) & (clean['salary'] <= up)]
print('Рядків після фільтрації викидів:', len(clean))
clean

Пропусків після заповнення: 0
Рядків після видалення дублікатів: 10
IQR-межі salary: [10000.0, 64000.0]
Рядків після фільтрації викидів: 9


,id,name,department,age,experience,salary
0,1,Anna,IT,25.0,2.0,30000.0
1,2,Bohdan,IT,31.0,6.0,45000.0
2,3,Olena,HR,30.0,4.0,28000.0
3,4,Petro,Sales,28.0,5.0,32000.0
4,5,Iryna,HR,35.0,10.0,40000.0
6,6,Oleh,IT,40.0,15.0,52000.0
7,7,Iryna,Sales,29.0,4.0,31000.0
8,8,Maksym,IT,33.0,8.0,40000.0
9,9,Yulia,IT,27.0,3.0,29000.0


Заповнення виконано до видалення дублікатів, а викид (зарплата 500 000) відфільтровано за IQR.

## Завдання 2. `sales.csv`: 5 пропусків у `money`, 3 дублікати, очищення

In [6]:
df = pd.read_csv(PATH)
print('Початковий розмір:', df.shape)

# штучні пропуски (5) у колонці money
rng = np.random.default_rng(0)
idx = rng.choice(df.index, size=5, replace=False)
df.loc[idx, 'money'] = np.nan
# три дублікати (копії перших трьох рядків)
df = pd.concat([df, df.iloc[:3]], ignore_index=True)
print('Після псування:', df.shape, '| пропусків money:', df['money'].isnull().sum(), '| дублікатів:', df.duplicated().sum())

Початковий розмір: (300, 6)
Після псування: (303, 6) | пропусків money: 5 | дублікатів: 3


In [7]:
df['money'] = df['money'].fillna(df['money'].mean())   # пропуски -> середнє
df = df.drop_duplicates()                              # видалення дублікатів
df['datetime'] = pd.to_datetime(df['datetime'])        # -> datetime64
print('Після очищення:', df.shape, '| пропусків money:', df['money'].isnull().sum(), '| дублікатів:', df.duplicated().sum())
print(df['datetime'].dtype)
df.to_csv(BASE + 'sales_cleaned_task2.csv', index=False)
print('Збережено: sales_cleaned_task2.csv')

Після очищення: (300, 6) | пропусків money: 0 | дублікатів: 0
datetime64[ns]
Збережено: sales_cleaned_task2.csv


## Завдання 3. Компоненти дати

In [8]:
df3 = pd.read_csv(PATH)
df3['datetime'] = pd.to_datetime(df3['datetime'])
df3['year'] = df3['datetime'].dt.year
df3['month'] = df3['datetime'].dt.month
df3['day'] = df3['datetime'].dt.day
df3['day_of_week'] = df3['datetime'].dt.dayofweek   # 0 = понеділок
df3['hour'] = df3['datetime'].dt.hour
df3[['datetime', 'year', 'month', 'day', 'day_of_week', 'hour']].head(10)

,datetime,year,month,day,day_of_week,hour
0,2025-03-01 09:27:40.117,2025,3,1,5,9
1,2025-03-01 09:40:11.474,2025,3,1,5,9
2,2025-03-01 10:12:26.345,2025,3,1,5,10
3,2025-03-01 10:38:22.609,2025,3,1,5,10
4,2025-03-01 11:33:02.443,2025,3,1,5,11
5,2025-03-01 13:25:01.486,2025,3,1,5,13
6,2025-03-01 15:05:10.917,2025,3,1,5,15
7,2025-03-01 16:11:12.944,2025,3,1,5,16
8,2025-03-01 16:17:59.602,2025,3,1,5,16
9,2025-03-01 17:37:31.632,2025,3,1,5,17


## Завдання 4. Обробка рядкових даних

In [9]:
cities = pd.DataFrame({'city': ['  Kyiv ', 'LVIV', 'Odesa  ', ' KHARKIV', 'kyiv '],
                       'phone': ['+38 (044) 111-22-33', '+38 (032) 222-33-44', '+38 (048) 333-44-55', '+38 (057) 444-55-66', '+38 (044) 111-22-33']})
print(cities, '\n')
cities['city'] = cities['city'].str.strip().str.lower()
cities['phone'] = cities['phone'].str.replace(r'[ ()\-]', '', regex=True)
print(cities)
print('\nУнікальних міст:', cities['city'].nunique())

       city                phone
0     Kyiv   +38 (044) 111-22-33
1      LVIV  +38 (032) 222-33-44
2   Odesa    +38 (048) 333-44-55
3   KHARKIV  +38 (057) 444-55-66
4     kyiv   +38 (044) 111-22-33 

      city          phone
0     kyiv  +380441112233
1     lviv  +380322223344
2    odesa  +380483334455
3  kharkiv  +380574445566
4     kyiv  +380441112233

Унікальних міст: 4


## Завдання 5. One-hot encoding для `cash_type`

In [10]:
enc = pd.get_dummies(pd.read_csv(PATH), columns=['cash_type'], dtype=int)
enc.head()

,date,datetime,card,money,coffee_name,cash_type_card,cash_type_cash
0,2025-03-01,2025-03-01 09:27:40.117,ANON-0000-0000-0001,35.99,Cappuccino,1,0
1,2025-03-01,2025-03-01 09:40:11.474,ANON-0000-0000-0002,35.99,Cocoa,1,0
2,2025-03-01,2025-03-01 10:12:26.345,NaN,24.00,Espresso,0,1
3,2025-03-01,2025-03-01 10:38:22.609,ANON-0000-0000-0004,38.70,Latte,1,0
4,2025-03-01,2025-03-01 11:33:02.443,ANON-0000-0000-0005,33.80,Americano with Milk,1,0


**Інтерпретація.** Замість однієї текстової колонки `cash_type` з'явилися дві бінарні: `cash_type_card` і `cash_type_cash`. У кожному рядку рівно одна з них дорівнює 1: `cash_type_card = 1` — оплата карткою, `cash_type_cash = 1` — готівкою. Такі числові ознаки придатні для алгоритмів машинного навчання.

## Завдання 6. Нормалізація та стандартизація `money`

In [11]:
d6 = pd.read_csv(PATH)
d6['money_norm'] = (d6['money'] - d6['money'].min()) / (d6['money'].max() - d6['money'].min())
d6['money_std'] = (d6['money'] - d6['money'].mean()) / d6['money'].std()
print('money_norm: min =', d6['money_norm'].min(), '| max =', d6['money_norm'].max())
print('money_std: mean =', d6['money_std'].mean(), '| std =', d6['money_std'].std())
print('Перевірка:',
      np.isclose(d6['money_norm'].min(), 0), np.isclose(d6['money_norm'].max(), 1),
      np.isclose(d6['money_std'].mean(), 0), np.isclose(d6['money_std'].std(), 1))

money_norm: min = 0.0 | max = 1.0
money_std: mean = -2.782959048393726e-16 | std = 1.0
Перевірка: True True True True


## Завдання 7. Ознака `is_card` через `np.where()`

In [12]:
d7 = pd.read_csv(PATH)
d7['is_card'] = np.where(d7['cash_type'] == 'card', 1, 0)
d7[['cash_type', 'is_card']].head(10)

,cash_type,is_card
0,card,1
1,card,1
2,cash,0
3,card,1
4,card,1
5,cash,0
6,card,1
7,card,1
8,card,1
9,card,1


## Відповіді на питання для самоперевірки

**1. Основні типи дефектів і чому їх усувають.** Пропуски, дублікати, викиди, неправильні типи даних, зайві пробіли/різний регістр, помилки введення. Вони спотворюють середнє, групування, сортування та моделі, тому висновки без очищення можуть бути хибними.

**2. Кількість пропусків у кожній колонці.** `df.isnull().sum()`.

**3. Обробка пропусків.** Видалити: `df.dropna(subset=['col'])`. Числові: `df['age'].fillna(df['age'].median())` (або `.mean()`). Категоріальні: `df['city'].fillna(df['city'].mode()[0])`.

**4. Повні дублікати.** `df.duplicated().sum()` — кількість, `df.drop_duplicates()` — видалення; за замовчуванням лишається перше входження (`keep='first'`). За окремими колонками: `drop_duplicates(subset=['id'])`.

**5. Викиди та IQR.** Викиди — значення, що значно відрізняються від основної маси. `IQR = Q3 − Q1`; межі: `Q1 − 1.5·IQR` і `Q3 + 1.5·IQR`; значення за межами — викиди.

**6. Чому дати мають тип `object`.** CSV зберігає все як текст, а Pandas не вгадує дати автоматично. Перетворення: `pd.to_datetime(df['col'], errors='coerce')`.

**7. Операції `.str`.** `str.strip()` — прибрати пробіли по краях, `str.lower()/upper()` — регістр, `str.replace('-', '')` — прибрати символи, `str.contains()`, `str.split()` тощо.

**8. Feature engineering.** Створення нових ознак з наявних. Через `.dt` отримують рік, місяць, день, день тижня, годину тощо — для аналізу сезонності, піків за годинами, будні/вихідні.

**9. One-hot encoding.** Кожна категорія стає окремою бінарною колонкою (`pd.get_dummies`): 1 — рядок належить категорії, 0 — ні. Дозволяє подавати категорії в алгоритми, що працюють лише з числами.

**10. Нормалізація проти стандартизації.** Нормалізація (min–max) переводить значення в [0, 1]: підтверджують `min = 0`, `max = 1`. Стандартизація дає середнє 0 і стандартне відхилення 1: підтверджують `mean ≈ 0`, `std ≈ 1`.

**11. Збереження без індексів.** `df.to_csv('file.csv', index=False)`.

**12. Чому важлива відтворюваність.** Зафіксовані в коді кроки дозволяють повторити очищення на нових даних, перевірити його коректність, знайти помилку та уникнути «ручних» невидимих правок.